# CatBoost hyperparameter tuning

In [1]:
# Load data

import pandas as pd
import numpy as np 

train_df = pd.read_parquet("data/train_data.parquet")

train_df.index = pd.to_numeric(train_df.index, errors="coerce")

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_df = train_df.drop(columns=cols_to_drop, errors="ignore")

In [2]:
# Separate features and targets

train_full_X = train_df.drop(["target", "target_annual_roi"], axis=1)
train_full_y_cat = train_df["target"]
train_full_y_reg = train_df["target_annual_roi"]

# Drop datetime features from the feature set

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_full_X = train_full_X.drop(columns=cols_to_drop, errors="ignore")

In [ ]:
# Create subsets of the data for different training sizes - chronological order is preserved, so we take the last N rows for each subset

train_1k_X = train_full_X.tail(1000)
train_1k_y_cat = train_full_y_cat.tail(1000)
train_1k_y_reg = train_full_y_reg.tail(1000)

train_10k_X = train_full_X.tail(10000)
train_10k_y_cat = train_full_y_cat.tail(10000)
train_10k_y_reg = train_full_y_reg.tail(10000)

train_100k_X = train_full_X.tail(100000)
train_100k_y_cat = train_full_y_cat.tail(100000)
train_100k_y_reg = train_full_y_reg.tail(100000)


cat_cols = train_df.select_dtypes(include=["object", "category"]).columns.tolist()

## Classification

[Parameters](https://catboost.ai/docs/en/references/training-parameters/common)

### 1k

In [4]:
# Check the dates of 1k subset to ensure all data is from same month

print(train_1k_X["issue_d_month"].unique())
print(train_1k_X["issue_d_year"].unique())

<IntegerArray>
[10]
Length: 1, dtype: Int64
<IntegerArray>
[2016]
Length: 1, dtype: Int64


In [5]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_cat.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_cat.head(200)

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 8), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 2.0, 10.0), # Class balancing weight for imbalanced target
        "objective": "Logloss", # Binary classification objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 3, 31) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostClassifier(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "scale_pos_weight",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_1k_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_1k_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_1k_parallel.html")


[I 2026-04-25 21:30:24,117] A new study created in memory with name: no-name-bc3d7201-1115-451a-9d1e-a0533284ca7f
[I 2026-04-25 21:30:31,386] Trial 3 finished with value: 0.5872045961701134 and parameters: {'iterations': 282, 'learning_rate': 0.012606038456206, 'depth': 2, 'l2_leaf_reg': 0.007450851774954155, 'random_strength': 0.31003771048613, 'rsm': 0.239721895051001, 'border_count': 122, 'scale_pos_weight': 9.328684651199282, 'boosting_type': 'Ordered', 'bootstrap_type': 'MVS', 'score_function': 'Cosine', 'subsample': 0.9518518058078105}. Best is trial 3 with value: 0.5872045961701134.
[I 2026-04-25 21:30:32,606] Trial 7 finished with value: 0.5778680514887412 and parameters: {'iterations': 250, 'learning_rate': 0.14655508418094432, 'depth': 2, 'l2_leaf_reg': 0.006800399529384482, 'random_strength': 2.0603373772466513, 'rsm': 0.6572999462201601, 'border_count': 145, 'scale_pos_weight': 9.72311324810007, 'boosting_type': 'Ordered', 'bootstrap_type': 'MVS', 'score_function': 'L2', 's


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 21:38:59,151] Trial 121 finished with value: 0.6603043029767167 and parameters: {'iterations': 896, 'learning_rate': 0.01552275543428587, 'depth': 3, 'l2_leaf_reg': 4.826150451390922, 'random_strength': 0.11304441093642684, 'rsm': 0.4907784909559726, 'border_count': 94, 'scale_pos_weight': 8.110101378821602, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 3.5063730186321385}. Best is trial 18 with value: 0.6740170461722186.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 21:39:01,719] Trial 113 finished with value: 0.6416574134677583 and parameters: {'iterations': 559, 'learning_rate': 0.027328862679486542, 'depth': 6, 'l2_leaf_reg': 3.0253425941715677, 'random_strength': 0.06264507865744205, 'rsm': 0.8395402656541269, 'border_count': 163, 'scale_pos_weight': 7.054806381693874, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bernoulli', 'score_function': 'L2', 'subsample': 0.3353513708301587}. Best is trial 18 with value: 0.6740170461722186.
[I 2026-04-25 21:39:04,764] Trial 119 finished with value: 0.6658972732248595 and parameters: {'iterations': 945, 'learning_rate': 0.015903985908878888, 'depth': 6, 'l2_leaf_reg': 4.631664089474147, 'random_strength': 0.10631758001597019, 'rsm': 0.3701916759283972, 'border_count': 95, 'scale_pos_weight': 7.354205051728528, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 3.5092149886096795}. Best is trial 18 with value: 0.6740170461722186.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 21:39:07,376] Trial 118 finished with value: 0.651271353167905 and parameters: {'iterations': 948, 'learning_rate': 0.015803638197727895, 'depth': 6, 'l2_leaf_reg': 2.978012446681382, 'random_strength': 0.09978175834451052, 'rsm': 0.4884420742471069, 'border_count': 68, 'scale_pos_weight': 8.072283756200141, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 3.5086476219499025}. Best is trial 18 with value: 0.6740170461722186.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 21:39:09,246] Trial 116 finished with value: 0.6260456784594717 and parameters: {'iterations': 778, 'learning_rate': 0.07800990899278011, 'depth': 6, 'l2_leaf_reg': 2.751914118733835, 'random_strength': 0.11113885130321743, 'rsm': 0.8315331427131063, 'border_count': 186, 'scale_pos_weight': 7.30274620308137, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 0.16333681083508989}. Best is trial 18 with value: 0.6740170461722186.
[I 2026-04-25 21:39:17,903] Trial 115 finished with value: 0.6544160150194633 and parameters: {'iterations': 910, 'learning_rate': 0.07592376155822776, 'depth': 6, 'l2_leaf_reg': 4.018360179592434, 'random_strength': 0.09420691161288067, 'rsm': 0.6776372765479879, 'border_count': 139, 'scale_pos_weight': 8.083620227054833, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 2.891440770233683}. Best is trial 18 with value: 0.6740170461722186.
[


BEST AUC: 0.6740
BEST PARAMETERS:
best_params = {
    "iterations": 433,
    "learning_rate": 0.03554345124418867,
    "depth": 4,
    "l2_leaf_reg": 9.840089950178873,
    "random_strength": 0.02805738171539979,
    "rsm": 0.9135819869398158,
    "border_count": 64,
    "scale_pos_weight": 6.980459859158197,
    "boosting_type": "Ordered",
    "bootstrap_type": "Bayesian",
    "score_function": "L2",
    "bagging_temperature": 9.907283731079449,
}

--- PARAMETER IMPORTANCE ---
  l2_leaf_reg         : 0.3015
  bootstrap_type      : 0.2216
  learning_rate       : 0.1397
  border_count        : 0.1314
  rsm                 : 0.0720
  scale_pos_weight    : 0.0567
  depth               : 0.0421
  iterations          : 0.0244
  random_strength     : 0.0090
  boosting_type       : 0.0017


In [6]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_catboost.best_params

best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "Logloss"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Cross-Val AUC: {study_catboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("="*40)

BEST PARAMS: {'iterations': 433, 'learning_rate': 0.03554345124418867, 'depth': 4, 'l2_leaf_reg': 9.840089950178873, 'random_strength': 0.02805738171539979, 'rsm': 0.9135819869398158, 'border_count': 64, 'scale_pos_weight': 6.980459859158197, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 9.907283731079449, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'Logloss', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}

Optuna Cross-Val AUC: 0.6740
Holdout Test AUC:     0.6839


### 10k

In [7]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_cat.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_cat.tail(2000)

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 8), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 2.0, 10.0), # Class balancing weight for imbalanced target
        "objective": "Logloss", # Binary classification objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False, # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 3, 31) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostClassifier(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "scale_pos_weight",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_10k_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_10k_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_10k_parallel.html")


[I 2026-04-25 21:40:20,180] A new study created in memory with name: no-name-912a2239-bb25-4f18-a36c-4f9f7ef42cf3
[I 2026-04-25 21:40:34,384] Trial 3 finished with value: 0.6974409884046421 and parameters: {'iterations': 287, 'learning_rate': 0.011814971919526544, 'depth': 4, 'l2_leaf_reg': 0.07175898100824094, 'random_strength': 0.14454483212723862, 'rsm': 0.4443830817525524, 'border_count': 215, 'scale_pos_weight': 7.17215171558566, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max_leaves': 23, 'min_data_in_leaf': 87, 'bagging_temperature': 5.1354656202952516}. Best is trial 3 with value: 0.6974409884046421.
[I 2026-04-25 21:40:35,239] Trial 0 finished with value: 0.6905105096058971 and parameters: {'iterations': 355, 'learning_rate': 0.01423492779191443, 'depth': 6, 'l2_leaf_reg': 0.15804993544737067, 'random_strength': 0.016553534710450433, 'rsm': 0.32037323851517185, 'border_count': 141, 'scale_pos_weight': 2.5755573131268408, 'boosting_type'


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 22:28:17,516] Trial 249 finished with value: 0.700308482540031 and parameters: {'iterations': 969, 'learning_rate': 0.005762996568664554, 'depth': 5, 'l2_leaf_reg': 0.08454062496819131, 'random_strength': 0.1889649575578709, 'rsm': 0.7450758430724499, 'border_count': 73, 'scale_pos_weight': 5.090348472831869, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max_leaves': 11, 'min_data_in_leaf': 30, 'bagging_temperature': 5.927507300585449}. Best is trial 150 with value: 0.7020147093561968.
[I 2026-04-25 22:28:19,409] Trial 247 finished with value: 0.6785471692221268 and parameters: {'iterations': 965, 'learning_rate': 0.005029476320771094, 'depth': 5, 'l2_leaf_reg': 8.051456490693678, 'random_strength': 0.1444654929806639, 'rsm': 0.7343380775461732, 'border_count': 74, 'scale_pos_weight': 5.119282638186606, 'boosting_type': 'Ordered', 'bootstrap_type': 'MVS', 'score_function': 'L2', 'subsample': 0.4843727018356229}. Best is trial 150 wit


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 22:28:32,854] Trial 245 finished with value: 0.6935351860945713 and parameters: {'iterations': 973, 'learning_rate': 0.005741552504863314, 'depth': 5, 'l2_leaf_reg': 0.08067428324222678, 'random_strength': 0.1738947996561404, 'rsm': 0.7317570284362788, 'border_count': 74, 'scale_pos_weight': 5.063724642174727, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 5.914060151608161}. Best is trial 150 with value: 0.7020147093561968.
[I 2026-04-25 22:28:53,877] Trial 253 finished with value: 0.6696425585881858 and parameters: {'iterations': 945, 'learning_rate': 0.005009374784671587, 'depth': 5, 'l2_leaf_reg': 0.0892760835266429, 'random_strength': 0.10159433759995821, 'rsm': 0.7621431554795705, 'border_count': 73, 'scale_pos_weight': 5.1751536557881765, 'boosting_type': 'Plain', 'bootstrap_type': 'MVS', 'grow_policy': 'Lossguide', 'max_leaves': 12, 'min_data_in_leaf': 28, 'subsample': 0.8738960674041869}. Best is trial 150


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 22:28:57,000] Trial 254 finished with value: 0.6690400564357966 and parameters: {'iterations': 946, 'learning_rate': 0.005045586709297658, 'depth': 5, 'l2_leaf_reg': 0.0820511965299804, 'random_strength': 0.050553104611132216, 'rsm': 0.768207628259473, 'border_count': 67, 'scale_pos_weight': 5.153013584021178, 'boosting_type': 'Plain', 'bootstrap_type': 'MVS', 'grow_policy': 'Lossguide', 'max_leaves': 12, 'min_data_in_leaf': 27, 'subsample': 0.8737575696070998}. Best is trial 150 with value: 0.7020147093561968.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 22:29:01,621] Trial 252 finished with value: 0.6686101311035791 and parameters: {'iterations': 975, 'learning_rate': 0.006482228257817107, 'depth': 5, 'l2_leaf_reg': 0.09501466077486698, 'random_strength': 0.05465788388738142, 'rsm': 0.7595203832930214, 'border_count': 66, 'scale_pos_weight': 5.0852925406651055, 'boosting_type': 'Ordered', 'bootstrap_type': 'MVS', 'score_function': 'L2', 'subsample': 0.5005966078642444}. Best is trial 150 with value: 0.7020147093561968.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST AUC: 0.7020
BEST PARAMETERS:
best_params = {
    "iterations": 954,
    "learning_rate": 0.006497925317522117,
    "depth": 6,
    "l2_leaf_reg": 0.0558959819268598,
    "random_strength": 0.0753981726868454,
    "rsm": 0.9249546627151901,
    "border_count": 183,
    "scale_pos_weight": 4.420312182485208,
    "boosting_type": "Plain",
    "bootstrap_type": "Bayesian",
    "grow_policy": "Lossguide",
    "max_leaves": 8,
    "min_data_in_leaf": 20,
    "bagging_temperature": 4.870558311628689,
}

--- PARAMETER IMPORTANCE ---
  iterations          : 0.3296
  learning_rate       : 0.2039
  bootstrap_type      : 0.1686
  depth               : 0.0763
  random_strength     : 0.0753
  scale_pos_weight    : 0.0506
  border_count        : 0.0449
  boosting_type       : 0.0262
  rsm                 : 0.0146
  l2_leaf_reg         : 0.0101


In [12]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_catboost.best_params.copy()

# Apply scaling trick


original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 2
best_params["learning_rate"] = original_lr / 2

print(f"[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params['iterations']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "Logloss"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Cross-Val AUC: {study_catboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("="*40)

[Scaling Trick Applied] CATBOOST: Trees 954 -> 1908, LR 0.0065 -> 0.0032
BEST PARAMS: {'iterations': 1908, 'learning_rate': 0.0032489626587610585, 'depth': 6, 'l2_leaf_reg': 0.0558959819268598, 'random_strength': 0.0753981726868454, 'rsm': 0.9249546627151901, 'border_count': 183, 'scale_pos_weight': 4.420312182485208, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max_leaves': 8, 'min_data_in_leaf': 20, 'bagging_temperature': 4.870558311628689, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'Logloss', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}

Optuna Cross-Val AUC: 0.7020
Holdout Test AUC:     0.6884


### 100k

In [13]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_cat.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_cat.tail(20000)

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 10), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-8, 100.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 2.0, 10.0), # Class balancing weight for imbalanced target
        "objective": "Logloss", # Binary classification objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False, # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 5, 512) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostClassifier(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "scale_pos_weight",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_100k_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_100k_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_100k_parallel.html")


[I 2026-04-25 22:36:16,542] A new study created in memory with name: no-name-8f7f73bb-21a7-4ba5-9617-f55581f59106
[I 2026-04-25 22:36:38,535] Trial 1 finished with value: 0.687080201567958 and parameters: {'iterations': 100, 'learning_rate': 0.011786159335307696, 'depth': 4, 'l2_leaf_reg': 0.0007079509186669808, 'random_strength': 6.215132838552753, 'rsm': 0.271697264637637, 'border_count': 213, 'scale_pos_weight': 8.421851667114398, 'boosting_type': 'Plain', 'bootstrap_type': 'Bernoulli', 'grow_policy': 'Depthwise', 'min_data_in_leaf': 19, 'score_function': 'Cosine', 'subsample': 0.2700140967854213}. Best is trial 1 with value: 0.687080201567958.
[I 2026-04-25 22:39:02,310] Trial 5 finished with value: 0.704707052164137 and parameters: {'iterations': 521, 'learning_rate': 0.028952147373694056, 'depth': 5, 'l2_leaf_reg': 2.2903231897548626e-07, 'random_strength': 0.07857635328013197, 'rsm': 0.7743655860120331, 'border_count': 139, 'scale_pos_weight': 9.421046698627677, 'boosting_type':


BEST AUC: 0.7114
BEST PARAMETERS:
best_params = {
    "iterations": 894,
    "learning_rate": 0.025302116751832124,
    "depth": 8,
    "l2_leaf_reg": 98.54551873139987,
    "random_strength": 0.030339567093011274,
    "rsm": 0.8587177152995968,
    "border_count": 185,
    "scale_pos_weight": 6.177887840704495,
    "boosting_type": "Ordered",
    "bootstrap_type": "Bernoulli",
    "score_function": "L2",
    "subsample": 0.7842935885515135,
}

--- PARAMETER IMPORTANCE ---
  scale_pos_weight    : 0.1942
  boosting_type       : 0.1825
  bootstrap_type      : 0.1804
  depth               : 0.1623
  learning_rate       : 0.1535
  iterations          : 0.0479
  rsm                 : 0.0346
  l2_leaf_reg         : 0.0215
  random_strength     : 0.0162
  border_count        : 0.0069


In [14]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_catboost.best_params.copy()

# Apply scaling trick
original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params['iterations']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "Logloss"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Cross-Val AUC: {study_catboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("="*40)


[Scaling Trick Applied] CATBOOST: Trees 894 -> 8940, LR 0.0253 -> 0.0025
BEST PARAMS: {'iterations': 8940, 'learning_rate': 0.0025302116751832124, 'depth': 8, 'l2_leaf_reg': 98.54551873139987, 'random_strength': 0.030339567093011274, 'rsm': 0.8587177152995968, 'border_count': 185, 'scale_pos_weight': 6.177887840704495, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bernoulli', 'score_function': 'L2', 'subsample': 0.7842935885515135, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'Logloss', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}

Optuna Cross-Val AUC: 0.7114
Holdout Test AUC:     0.7148


### Full data set

In [15]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*4
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_cat[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_cat[split_index:]

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 10000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 10), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-8, 100.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 2.0, 10.0), # Class balancing weight for imbalanced target
        "objective": "Logloss", # Binary classification objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False, # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 5, 512) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostClassifier(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "scale_pos_weight",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_full_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_full_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_full_parallel.html")


[I 2026-04-26 02:11:03,223] A new study created in memory with name: no-name-3cd54803-c19e-41b1-b438-726a03124e2a
[I 2026-04-26 02:25:03,084] Trial 2 finished with value: 0.7224281361931979 and parameters: {'iterations': 396, 'learning_rate': 0.005117172272136172, 'depth': 4, 'l2_leaf_reg': 1.2277085762690914e-08, 'random_strength': 0.007157960792504025, 'rsm': 0.8743685831355734, 'border_count': 207, 'scale_pos_weight': 3.608353011786745, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'SymmetricTree', 'score_function': 'Cosine', 'bagging_temperature': 4.709290614061601}. Best is trial 2 with value: 0.7224281361931979.
[I 2026-04-26 03:05:24,557] Trial 6 finished with value: 0.7277020991780146 and parameters: {'iterations': 391, 'learning_rate': 0.00877558504884166, 'depth': 10, 'l2_leaf_reg': 22.387166535056743, 'random_strength': 0.2088257980170935, 'rsm': 0.955431534391497, 'border_count': 99, 'scale_pos_weight': 7.883493226709827, 'boosting_type': 'Plain', '

KeyboardInterrupt: 

In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_catboost.best_params.copy()

# Apply scaling trick


original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params['iterations']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "Logloss"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Cross-Val AUC: {study_catboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("="*40)

## Regression

[Parameters](https://catboost.ai/docs/en/references/training-parameters/common)

### 1k

In [16]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_reg.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_reg.head(200)

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 8), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "objective": "RMSE", # Regression objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 3, 31) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostRegressor(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_1k_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_1k_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_1k_parallel.html")


[I 2026-04-26 09:35:05,085] A new study created in memory with name: no-name-428bbc8f-82dc-4583-a4f1-8efe0902bb16
[I 2026-04-26 09:35:06,788] Trial 1 finished with value: 0.3206080735305823 and parameters: {'iterations': 260, 'learning_rate': 0.05764775623977107, 'depth': 2, 'l2_leaf_reg': 1.9588073144766103, 'random_strength': 0.0034399568829632135, 'rsm': 0.2838689946143098, 'border_count': 134, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 2.890491659130583}. Best is trial 1 with value: 0.3206080735305823.
[I 2026-04-26 09:35:08,642] Trial 5 finished with value: 0.31825801262520625 and parameters: {'iterations': 216, 'learning_rate': 0.012761438892139924, 'depth': 3, 'l2_leaf_reg': 0.468539643734418, 'random_strength': 0.027204782574556764, 'rsm': 0.9722745523370238, 'border_count': 253, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max_leaves': 4, 'min_data_in_leaf': 87, 'bagging_tempe


BEST RMSE: 0.3144
BEST PARAMETERS:
best_params = {
    "iterations": 976,
    "learning_rate": 0.00976818825318037,
    "depth": 4,
    "l2_leaf_reg": 0.027265370477102587,
    "random_strength": 0.7031697546586838,
    "rsm": 0.8385396695909164,
    "border_count": 201,
    "boosting_type": "Ordered",
    "bootstrap_type": "Bayesian",
    "score_function": "Cosine",
    "bagging_temperature": 9.530476491737797,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.9508
  iterations          : 0.0140
  bootstrap_type      : 0.0125
  rsm                 : 0.0085
  border_count        : 0.0082
  boosting_type       : 0.0026
  random_strength     : 0.0014
  depth               : 0.0013
  l2_leaf_reg         : 0.0008


In [17]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_catboost.best_params

best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "RMSE"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Cross-Val RMSE: {study_catboost.best_value:.4f}")
print(f"Holdout Test RMSE:     {holdout_rmse:.4f}")
print("="*40)

BEST PARAMS: {'iterations': 976, 'learning_rate': 0.00976818825318037, 'depth': 4, 'l2_leaf_reg': 0.027265370477102587, 'random_strength': 0.7031697546586838, 'rsm': 0.8385396695909164, 'border_count': 201, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 9.530476491737797, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'RMSE', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}

Optuna Cross-Val RMSE: 0.3144
Holdout Test RMSE:     0.3340


### 10k

In [18]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_reg.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_reg.tail(2000)

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 8), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "objective": "RMSE", # Regression objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False, # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 3, 31) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostRegressor(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_10k_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_10k_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_10k_parallel.html")


[I 2026-04-26 10:05:54,557] A new study created in memory with name: no-name-4c4da6da-5cca-4dc7-b6c2-205bdd9b0da0
[I 2026-04-26 10:06:10,100] Trial 2 finished with value: 0.31121998114057103 and parameters: {'iterations': 859, 'learning_rate': 0.1749200343284882, 'depth': 3, 'l2_leaf_reg': 0.0036528914748255453, 'random_strength': 0.7634974526460588, 'rsm': 0.37865276175104623, 'border_count': 152, 'boosting_type': 'Plain', 'bootstrap_type': 'MVS', 'grow_policy': 'SymmetricTree', 'score_function': 'L2', 'subsample': 0.3157131881566521}. Best is trial 2 with value: 0.31121998114057103.
[I 2026-04-26 10:06:17,167] Trial 7 finished with value: 0.2979299940087425 and parameters: {'iterations': 611, 'learning_rate': 0.02065640778824637, 'depth': 2, 'l2_leaf_reg': 2.0431013480789515, 'random_strength': 1.0124176312513955, 'rsm': 0.7573138246806337, 'border_count': 211, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max_leaves': 22, 'min_data_in_leaf': 35


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 10:34:50,733] Trial 140 finished with value: 0.29700052068038396 and parameters: {'iterations': 908, 'learning_rate': 0.012763956553074614, 'depth': 4, 'l2_leaf_reg': 0.3088398529333473, 'random_strength': 2.68677326425155, 'rsm': 0.744147359668745, 'border_count': 103, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 7.545412966046143}. Best is trial 40 with value: 0.2966091160106081.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 10:34:51,296] Trial 138 finished with value: 0.29710364822773117 and parameters: {'iterations': 908, 'learning_rate': 0.012419620505449983, 'depth': 5, 'l2_leaf_reg': 0.28556984446035416, 'random_strength': 4.932583605491949, 'rsm': 0.5544106827664926, 'border_count': 107, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 5.363044521531694}. Best is trial 40 with value: 0.2966091160106081.
[I 2026-04-26 10:34:58,815] Trial 141 finished with value: 0.29704943717356647 and parameters: {'iterations': 977, 'learning_rate': 0.012533188328160407, 'depth': 4, 'l2_leaf_reg': 0.2823093271389073, 'random_strength': 4.663953365628512, 'rsm': 0.7488369395614154, 'border_count': 105, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 5.340680518623314}. Best is trial 40 with value: 0.2966091160106081.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 10:35:09,286] Trial 142 finished with value: 0.29691641849952805 and parameters: {'iterations': 907, 'learning_rate': 0.014650464000978436, 'depth': 4, 'l2_leaf_reg': 0.2713179363210563, 'random_strength': 3.8327513231803443, 'rsm': 0.744867101720851, 'border_count': 153, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 6.112443024703257}. Best is trial 40 with value: 0.2966091160106081.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 10:35:23,165] Trial 139 finished with value: 0.2968049791601305 and parameters: {'iterations': 969, 'learning_rate': 0.01460958035398821, 'depth': 5, 'l2_leaf_reg': 0.19832002184547623, 'random_strength': 0.3567191173008058, 'rsm': 0.738050846104574, 'border_count': 103, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 5.356827165173819}. Best is trial 40 with value: 0.2966091160106081.
[I 2026-04-26 10:37:15,984] Trial 13 finished with value: 0.7359240728067847 and parameters: {'iterations': 4897, 'learning_rate': 0.016370061296226476, 'depth': 6, 'l2_leaf_reg': 0.2727482933935999, 'random_strength': 3.5964693066900386, 'rsm': 0.3821606351295279, 'border_count': 167, 'scale_pos_weight': 4.733863711713026, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 9.125809324314888}. Best is trial 13 with value: 0.7359240728067847.
[I 2026-04-26 10:37:45,594] Trial 13


BEST RMSE: 0.2966
BEST PARAMETERS:
best_params = {
    "iterations": 544,
    "learning_rate": 0.027650691306981594,
    "depth": 5,
    "l2_leaf_reg": 0.5148645656737443,
    "random_strength": 0.1893828349911231,
    "rsm": 0.7999433598790255,
    "border_count": 194,
    "boosting_type": "Ordered",
    "bootstrap_type": "Bayesian",
    "score_function": "L2",
    "bagging_temperature": 8.170909952414544,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.8100
  bootstrap_type      : 0.0861
  rsm                 : 0.0309
  iterations          : 0.0254
  l2_leaf_reg         : 0.0204
  border_count        : 0.0087
  boosting_type       : 0.0085
  depth               : 0.0063
  random_strength     : 0.0037


In [19]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_catboost.best_params.copy()

# Apply scaling trick


original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 2
best_params["learning_rate"] = original_lr / 2

print(f"\n[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params['iterations']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "RMSE"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Cross-Val RMSE: {study_catboost.best_value:.4f}")
print(f"Holdout Test RMSE:     {holdout_rmse:.4f}")
print("="*40)


[Scaling Trick Applied] CATBOOST: Trees 544 -> 1088, LR 0.0277 -> 0.0138
BEST PARAMS: {'iterations': 1088, 'learning_rate': 0.013825345653490797, 'depth': 5, 'l2_leaf_reg': 0.5148645656737443, 'random_strength': 0.1893828349911231, 'rsm': 0.7999433598790255, 'border_count': 194, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 8.170909952414544, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'RMSE', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}

Optuna Cross-Val RMSE: 0.2966
Holdout Test RMSE:     0.3122


### 100k

In [20]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning


X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_reg.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_reg.tail(20000)

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 10), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-8, 100.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "objective": "RMSE", # Regression objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False, # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 5, 512) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostRegressor(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_100k_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_100k_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_100k_parallel.html")


[I 2026-04-26 10:38:01,962] A new study created in memory with name: no-name-92b41a08-583e-4d57-a1ba-70a8b359d5af
[I 2026-04-26 10:39:09,495] Trial 2 finished with value: 0.3024753664951711 and parameters: {'iterations': 324, 'learning_rate': 0.006994633234659066, 'depth': 5, 'l2_leaf_reg': 7.690569983928778e-08, 'random_strength': 0.20204959872204886, 'rsm': 0.35533999411198103, 'border_count': 64, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bernoulli', 'score_function': 'L2', 'subsample': 0.2594852262145646}. Best is trial 2 with value: 0.3024753664951711.
[I 2026-04-26 10:39:59,681] Trial 7 finished with value: 0.300819159021708 and parameters: {'iterations': 324, 'learning_rate': 0.18525489468588302, 'depth': 2, 'l2_leaf_reg': 1.0389424660506077, 'random_strength': 0.0011900467641136516, 'rsm': 0.8297319520832989, 'border_count': 92, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 5.994185547847424}. Best is trial 7 with v


BEST RMSE: 0.2993
BEST PARAMETERS:
best_params = {
    "iterations": 993,
    "learning_rate": 0.010876805260096451,
    "depth": 9,
    "l2_leaf_reg": 25.36922452973143,
    "random_strength": 1.1801448166458588,
    "rsm": 0.7742077921478168,
    "border_count": 217,
    "boosting_type": "Plain",
    "bootstrap_type": "MVS",
    "grow_policy": "Depthwise",
    "min_data_in_leaf": 299,
    "score_function": "L2",
    "subsample": 0.269182776220681,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.6736
  depth               : 0.1173
  bootstrap_type      : 0.0564
  border_count        : 0.0458
  boosting_type       : 0.0346
  l2_leaf_reg         : 0.0251
  iterations          : 0.0227
  rsm                 : 0.0131
  random_strength     : 0.0113


In [21]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_catboost.best_params.copy()

# Apply scaling trick

original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params['iterations']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "RMSE"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Cross-Val RMSE: {study_catboost.best_value:.4f}")
print(f"Holdout Test RMSE:     {holdout_rmse:.4f}")
print("="*40)


[Scaling Trick Applied] CATBOOST: Trees 993 -> 9930, LR 0.0109 -> 0.0011
BEST PARAMS: {'iterations': 9930, 'learning_rate': 0.001087680526009645, 'depth': 9, 'l2_leaf_reg': 25.36922452973143, 'random_strength': 1.1801448166458588, 'rsm': 0.7742077921478168, 'border_count': 217, 'boosting_type': 'Plain', 'bootstrap_type': 'MVS', 'grow_policy': 'Depthwise', 'min_data_in_leaf': 299, 'score_function': 'L2', 'subsample': 0.269182776220681, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'RMSE', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}

Optuna Cross-Val RMSE: 0.2993
Holdout Test RMSE:     0.2984


### Full data set

In [ ]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*4
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_reg[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_reg[split_index:]

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 10000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 10), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-8, 100.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "objective": "RMSE", # Regression objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False, # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 5, 512) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostRegressor(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_full_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_full_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_full_parallel.html")


[I 2026-04-23 17:17:31,555] A new study created in memory with name: no-name-22cdb8b3-7ba1-4f27-85c5-a03cfe7c6f97


In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_catboost.best_params.copy()

# Apply scaling trick


original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params['iterations']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "RMSE"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Cross-Val RMSE: {study_catboost.best_value:.4f}")
print(f"Holdout Test RMSE:     {holdout_rmse:.4f}")
print("="*40)